# ProQuest v5: reviewer search and coding rerun

**Step 0: run your own search first.** Reproduce `search/topical_query.txt` using `search/REVIEWER_SEARCH.md`; export **your new search's** XML and Extended Metadata CSV into this authorized TDM Studio Workbench. Record their paths, search time and interface filters; the displayed result count is optional provenance. The package contains scripts, configuration and instructions, with no raw TDM records or prebuilt GOID list. A fresh search creates a new rerun, not the historical evidence identity. Read README.md and LICENSED_INPUTS.md. All actions that process licensed records or call the model are disabled by default.

**Integrated single scope pass:** use [the operative codebook](config/coding/dissertation_codebook_v5_rev15_reproduction.md). The first scope response assesses all aims and S1–S5, including endogenous S4. Supported uncertainty is final scope coding; content returns C1–C9 only for fixed targets. Use a private plan/output folder bound to this configuration.


In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd().resolve()
assert (ROOT / 'manifest/files.json').is_file(), 'Open this notebook from the extracted package folder'
sys.path.insert(0, str(ROOT / 'src/coding'))
import public_workflow as workflow
from run_v5_rev14_hybrid_production import RUNNER_VERSION, _classify_http400_response
assert RUNNER_VERSION == 'v5_rev15_single_scope_public_rerun_v1' and callable(_classify_http400_response)
print(workflow.verify_package())


## 1. Prepare from your own replicated search

Set both paths to **your new search's** files in this Workbench. Also record its execution time and exact interface filters. The displayed result count is optional provenance and is never compared with the exports. The Extended Metadata CSV must include GOID, Title, Authors/Author, Date, Degree Date, Language, School Location and Source Type. The script matches XML and metadata by GOID, applies the documented document, date, language and U.S. location rules, deduplicates by normalized title–author–year, freezes the private evidence and assignment, then runs a no-call preflight. Search results vary, so counts and missing values never stop preparation: a record without matching evidence, or without the metadata a rule needs, is excluded with a recorded reason, and title differences or missing authors are flagged. Preparation stops only if a required column is absent or no assessable record remains. Review `private/reviewer_rerun/search_audit_PRIVATE.json`, `search_removals_PRIVATE.csv`, `search_review_flags_PRIVATE.csv` and `source_partition/unassessable_goids.json` before paying for live coding. Keep all generated files inside TDM Studio.


In [ ]:
PREPARE_FROM_MY_SEARCH = False
XML_DIR = None       # Replace with Path('/home/ec2-user/SageMaker/my_new_search/xml')
METADATA_CSV = None  # Replace with Path('/home/ec2-user/SageMaker/my_new_search/Extended_Metadata.csv')
SEARCH_EXECUTED_AT = None  # Your search execution date/time, as shown by Studio
INTERFACE_FILTERS = None  # All interface filters you selected; describe them exactly
SEARCH_RESULT_COUNT = None  # Optional: count shown by Studio, kept as provenance only
RERUN = ROOT / 'private/reviewer_rerun'
WORKERS = 1
WORKER_ID = 1
if PREPARE_FROM_MY_SEARCH:
    assert XML_DIR and METADATA_CSV and SEARCH_EXECUTED_AT and INTERFACE_FILTERS
    if not RERUN.exists():
        print(workflow.prepare_from_search(XML_DIR, METADATA_CSV, RERUN, workers=WORKERS,
              search_executed_at=SEARCH_EXECUTED_AT, interface_filters=INTERFACE_FILTERS,
              search_result_count=SEARCH_RESULT_COUNT))
    workflow.extract(XML_DIR, RERUN, WORKER_ID)
    workflow.run(RERUN, WORKER_ID, mode='preflight')
else:
    print('Preparation disabled; run your own search first')


## 2. Run scope, then content

Verify access to both requested model snapshots in your authorized TDM Studio session before processing records. After the first few hundred scope responses, project cost from `usage_by_stage` in `worker_1_output/production_progress.json` (see README.md). Set RUN_LIVE=True for scope. After scope completes, inspect private progress and failures; set RELEASE_CONTENT=True and rerun this cell for content. The HTTP 400 record guard is built into both live stages. Confirmed policy refusals are explicitly deferred for later review; configuration failures still stop. Keep the same plan and checkpoint on resume. A deliberate one-pass failure retry is optional. See LICENSED_INPUTS.md.


In [ ]:
RUN_LIVE = False
RELEASE_CONTENT = False
RETRY_FAILURES_ONCE = False
BASE_URL = None  # Authorized TDM proxy endpoint
KEY_FILE = None  # Workbench private key-file path; never paste key text
PACE_TPM = None  # Measured authorized pacing
if RUN_LIVE:
    workflow.run(RERUN, WORKER_ID, mode='live', release_content=RELEASE_CONTENT,
                 base_url=BASE_URL, key_file=KEY_FILE, pace_tpm=PACE_TPM,
                 retry_failures_once=RETRY_FAILURES_ONCE)
else:
    print('Live model calls disabled')


## 3. Reconcile the completed rerun

Enable only after all workers finish. This creates private record-level accounting. Deferred failures remain failures and cannot be reported as coded undetermined records. A standard final merge requires complete, validated exports.


In [ ]:
MERGE_COMPLETE = False
if MERGE_COMPLETE:
    print(workflow.merge(RERUN, synthetic=False))
    print(workflow.account(RERUN))
else:
    print('Merge disabled until complete')
